# 06. 최종 학습과 분류 결과

03~05 에서 고른 설정으로 **본 학습**을 하고, **여기서 처음으로 테스트셋을 연다.**

> 지금까지 모든 결정(전처리·모델·학습률·해상도·정규화·배치)은 검증셋으로만 했다.
> 테스트셋을 보면서 고르면 그 점수는 더 이상 "처음 보는 데이터의 성능"이 아니다.

| 절 | 내용 |
|---|---|
| 6-1 | 최종 설정 확인 |
| 6-2 | 본 학습 (에폭 충분히 + 조기종료 + 코사인 스케줄) |
| 6-3 | 테스트셋 최종 평가 + 신뢰구간 |
| 6-4 | 혼동행렬 — 어디서 틀리나 |
| 6-5 | 틀린 사례 관찰 |

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from PIL import Image

import wbc
wbc.use_korean_font()
wbc.NUM_WORKERS = 4

cfg = wbc.load_cfg()
print('최종 설정')
print(json.dumps(cfg, ensure_ascii=False, indent=1))

## 6-2. 본 학습

스크리닝은 6 에폭이었지만 여기서는 **최대 25 에폭**을 허용하고 조기종료에 맡긴다.

`scheduler='cosine'` 을 쓴다. 학습 후반에 학습률을 부드럽게 낮춰 마지막 성능을 짜내는 방법이다.
(스크리닝 때는 `ReduceLROnPlateau` 로 안정성을 우선했다)

In [ ]:
FINAL_RUN = 'FINAL'
res = wbc.run_experiment(FINAL_RUN,
                         model_name=cfg['model_name'], preset=cfg['preset'],
                         image_size=cfg['image_size'], batch_size=cfg['batch_size'],
                         lr=cfg['lr'], weight_decay=cfg['weight_decay'],
                         label_smoothing=cfg['label_smoothing'], dropout=cfg['dropout'],
                         epochs=cfg['full_epochs'], patience=6, seed=42, scheduler='cosine')
if res: wbc.plot_history(res['history'], '최종 모델'); plt.show()

In [ ]:
h = wbc.runs_table()
row = h[h.run_id == FINAL_RUN].iloc[0]
print(f"best_epoch {int(row.best_epoch)} / 최대 {cfg['full_epochs']}   "
      f"에폭당 {row.epoch_sec:.0f}초 (예산 180초)"
      + ('   ⚠ 예산 초과' if row.epoch_sec > 180 else '   ○ 조건 충족'))
print(f"검증 정확도 {row.val_accuracy:.4f} / macro-F1 {row.val_macro_f1:.4f}")

## 6-3. 테스트셋 최종 평가

검증 macro-F1 이 가장 좋았던 시점의 **저장된 가중치**를 불러와 평가한다.
테스트셋은 모델 선택에 전혀 개입하지 않았다.

In [ ]:
model, m_test = wbc.eval_on_test(FINAL_RUN)     # 예측 확률을 npz 로 저장 -> 08 가설검정에서 쓴다
print(wbc.summarize(m_test, '최종 모델 TEST'))
print()
for metric in ['accuracy', 'macro_f1', 'balanced']:
    ci = wbc.bootstrap_ci(m_test['trues'], m_test['preds'], metric=metric, n_boot=2000)
    print(f"  {metric:9s} {ci['point']:.4f}   95% 신뢰구간 [{ci['lo']:.4f}, {ci['hi']:.4f}]")

### 신뢰구간을 반드시 붙인다

"정확도 0.9832" 는 테스트셋 2,487장에서 잰 **하나의 추정값**이고, 표본이 바뀌면 값도 흔들린다.
부트스트랩 신뢰구간이 그 흔들림의 폭을 말해준다.

두 모델의 신뢰구간이 크게 겹치면 "더 좋다"고 단정할 수 없다 → **08 의 검정이 필요한 이유.**

> 그리고 이 숫자를 최종 성능이라고 단정하지 않는다.
> 01-5 에서 본 대로 원본 단위 분리가 보장되지 않으므로,
> **08 가설검정 3** 에서 증강되지 않은 원본으로 한 번 더 잰다.

## 6-4. 혼동행렬 — 어디서 틀리나

In [ ]:
wbc.plot_confusion(m_test['trues'], m_test['preds'], title='최종 모델 — 혼동행렬 (개수)'); plt.show()
wbc.plot_confusion(m_test['trues'], m_test['preds'], normalize=True,
                   title='최종 모델 — 혼동행렬 (행 정규화 = 클래스별 재현율)'); plt.show()
display(wbc.class_report(m_test))

In [ ]:
# 어떤 쌍이 가장 많이 헷갈리나
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(m_test['trues'], m_test['preds'])
pairs = [(wbc.CLASS_NAMES[i], wbc.CLASS_NAMES[j], int(cm[i, j]))
         for i in range(4) for j in range(4) if i != j]
pairs.sort(key=lambda x: -x[2])
print('가장 많이 혼동한 쌍 (정답 -> 예측)')
for a, b, n in pairs[:5]:
    print(f'  {a:11s} -> {b:11s} : {n}장')

### 읽는 법

- 대각선이 각 클래스의 **재현율**이다. 특정 클래스만 낮으면 그 클래스가 어렵다는 뜻이다.
- 백혈구 분류에서 가장 헷갈리기 쉬운 쌍은 **EOSINOPHIL ↔ NEUTROPHIL** 이다.
  둘 다 핵이 여러 덩이로 갈라진 과립구라 형태가 비슷하고,
  구분 근거는 세포질 과립의 색(호산구는 분홍/주황)이다.
- 이 쌍의 오분류가 많다면 → **"색이 중요한 신호"** 라는 01-3 의 관찰과 일치하고,
  03 에서 강한 색 증강이 불리했던 것도 같은 이유로 설명된다.

**혼동행렬에서 찾은 패턴을 07 의 CAM 으로 확인한다.**

## 6-5. 틀린 사례 관찰

In [ ]:
_, ds_test = wbc.make_eval_loader(os.path.join(wbc.DATA_DIR, 'TEST'), image_size=int(cfg['image_size']))
wrong = np.where(m_test['preds'] != m_test['trues'])[0]
print(f'틀린 것 {len(wrong)}장 / 전체 {len(m_test["trues"])}장')

if len(wrong):
    sel = np.random.RandomState(0).choice(wrong, min(8, len(wrong)), replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(13, 6)); axes = axes.ravel()
    for ax, i in zip(axes, sel):
        x, y = ds_test[int(i)]
        p = m_test['preds'][i]; conf = m_test['probs'][i][p]
        ax.imshow(wbc._to_numpy_img(x)); ax.axis('off')
        ax.set_title(f'정답 {wbc.CLASS_NAMES[y][:4]} / 예측 {wbc.CLASS_NAMES[p][:4]} ({conf:.2f})', fontsize=10)
    for ax in axes[len(sel):]: ax.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# 확신도 분포 — 틀릴 때 모델이 확신하고 있었나?
conf = m_test['probs'].max(1)
ok = m_test['preds'] == m_test['trues']
plt.figure(figsize=(6.5, 3.6))
plt.hist(conf[ok], bins=30, alpha=.6, label='맞힌 경우')
plt.hist(conf[~ok], bins=30, alpha=.6, label='틀린 경우')
plt.xlabel('예측 확신도 (softmax 최댓값)'); plt.ylabel('장수'); plt.legend(); plt.grid(alpha=.3)
plt.title('확신도 분포'); plt.show()
print(f'맞힌 경우 평균 확신도 {conf[ok].mean():.3f} / 틀린 경우 {conf[~ok].mean():.3f}')

**틀릴 때 확신도가 낮다면** 좋은 신호다. 임계값을 두고 "확신 없으면 사람에게 넘긴다"는
운용이 가능하기 때문이다. **틀리면서도 확신도가 높다면** 모델이 잘못된 단서를 배웠을 가능성이 있고,
07 의 CAM 으로 그 단서가 무엇인지 볼 수 있다.

## 06 정리

- 최종 설정으로 본 학습을 마치고, 테스트셋을 딱 한 번 열어 평가했다
- 정확도·macro-F1·클래스별 성능·신뢰구간·혼동행렬·오분류 사례를 확보했다
- 예측 확률을 `results/preds/FINAL_TEST.npz` 에 저장했다 (08 가설검정용)

→ 다음: **07_CAM_시각화.ipynb**